# Question Answering Over a Document

A foundation model only knows what it learned in training. To answer questions about *your* documents, you give it the relevant context in the request. The Converse API makes this easy with a **document content block**: attach a file and Bedrock parses it for the model.

This notebook shows:
1. Asking without the document (the model can't know)
2. Asking with the document attached (grounded answer)
3. Referencing a document in Amazon S3 instead of sending bytes

> **Teaching/Learning Tip:** This is the core idea behind RAG (retrieval-augmented generation). RAG just automates the "find the relevant document" step; here we supply it directly.

## Setup

The sample PDF is a made-up quantum computing report - it contains facts the model could not have seen in training (invented vendors and numbers). That's deliberate: it lets us prove the answer comes from the document, not the model's memory.

In [ ]:
import boto3

client = boto3.client("bedrock-runtime", region_name="us-east-1")
MODEL_ID = "us.amazon.nova-lite-v1:0"

QUESTION = (
    "According to the report, which vendor has the largest market share, "
    "and what is the primary bottleneck for the industry?"
)

with open("sample_report.pdf", "rb") as f:
    doc_bytes = f.read()

print(f"Loaded {len(doc_bytes)} bytes of PDF")

## 1. Ask WITHOUT the document

First, ask the question with no context. The model has never seen this report, so it can only guess or admit it doesn't know.

In [ ]:
response = client.converse(
    modelId=MODEL_ID,
    messages=[{"role": "user", "content": [{"text": QUESTION}]}],
    inferenceConfig={"maxTokens": 300, "temperature": 0.3},
)
print(response["output"]["message"]["content"][0]["text"])

## 2. Ask WITH the document attached

Now add a `document` content block before the question. Bedrock parses the PDF and the model answers from its actual contents.

> **Teaching/Learning Tip:** The content list can mix block types. Here it's `[document, text]` - the document provides context, the text asks the question.

In [ ]:
response = client.converse(
    modelId=MODEL_ID,
    messages=[{
        "role": "user",
        "content": [
            {"document": {
                "format": "pdf",
                "name": "QuantumReport",
                "source": {"bytes": doc_bytes},
            }},
            {"text": QUESTION},
        ],
    }],
    inferenceConfig={"maxTokens": 300, "topP": 0.1, "temperature": 0.3},
)
print(response["output"]["message"]["content"][0]["text"])

Notice the difference: with the document attached, the answer cites the report's actual vendors and bottleneck. That's grounding.

## 3. Reference a document in Amazon S3

For large documents, you don't want to ship the bytes through your app on every call. Instead, put the file in S3 and point Converse at it with an `s3Location`. The message shape is identical except for the `source`:

```python
"source": {
    "s3Location": {
        "uri": "s3://your-bucket/documents/sample_report.pdf",
        "bucketOwner": "123456789012",
    }
}
```

The runnable version (with bucket create/upload and cleanup) is in `qa_s3.py`:

```bash
python qa_s3.py            # create bucket, upload PDF, ask the question
python qa_s3.py --cleanup  # delete the bucket when done
```

> **Teaching/Learning Tip:** bytes vs. S3 is a size/convenience tradeoff. Small one-off file? Send bytes. Large or reused document? Store it in S3 and reference it - and remember to clean the bucket up afterward.